In [ ]:
import os
import pandas as pd
import pyodbc

SERVER = "10.147.17.185"
PORT = 1433
USERNAME = "cmpcuser"
PASSWORD = os.getenv("OTMS_SQL_PASSWORD_DEV")
DRIVER = "{ODBC Driver 17 for SQL Server}"

DB = "cmpc_20240925_093000"
SCHEMA = "dbo"
TABLE = "ypf_alarms"

conn_str = (
    f"DRIVER={DRIVER};"
    f"SERVER={SERVER},{PORT};"
    f"DATABASE={DB};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "TrustServerCertificate=yes;"
)
conn = pyodbc.connect(conn_str)
print("Conectado.")

Conectado.


In [ ]:
DAYS_WINDOW = 30

q = f"""
WITH mx AS (
  SELECT MAX(ALARMDATETIME) AS mx_t
  FROM [{DB}].[{SCHEMA}].[{TABLE}]
),
minute_counts AS (
  SELECT
    DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0) AS [minute],
    COUNT(*) AS alarms
  FROM [{DB}].[{SCHEMA}].[{TABLE}] a
  CROSS JOIN mx
  WHERE a.ALARMDATETIME >= DATEADD(day, -{DAYS_WINDOW}, mx.mx_t)
  GROUP BY DATEADD(minute, DATEDIFF(minute, 0, a.ALARMDATETIME), 0)
)
SELECT alarms
FROM minute_counts;
"""

df_min = pd.read_sql(q, conn)
x = df_min["alarms"].to_numpy()

rate_p95  = float(pd.Series(x).quantile(0.95))
rate_p99  = float(pd.Series(x).quantile(0.99))
rate_p999 = float(pd.Series(x).quantile(0.999))
max_rate  = int(x.max())

baseline = {
    "window_days": DAYS_WINDOW,
    "rate_p95": rate_p95,
    "rate_p99": rate_p99,
    "rate_p999": rate_p999,
    "max_per_minute": max_rate,
}
baseline

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_21220\3620823964.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_min = pd.read_sql(q, conn)


{'window_days': 30,
 'rate_p95': 77.25,
 'rate_p99': 144.0,
 'rate_p999': 411.22499999999945,
 'max_per_minute': 540}

In [ ]:
df = pd.read_csv("artifacts/blocks_features.csv", parse_dates=["start","end"])

# Si tu df ya tiene prio1_share, perfecto. Si no, lo calculamos luego.
df.columns

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/blocks_features.csv'